# 04 - Retrieval híbrido

Esta etapa constrói o retrieval compartilhado por todos os LLMs.

Componentes:

1. embeddings locais com Sentence Transformers;
2. BM25 lexical;
3. fusão por Reciprocal Rank Fusion, RRF;
4. filtro temporal aplicado **antes** do ranking histórico.

O mesmo retrieval será usado para todos os modelos do experimento.

In [1]:
import pandas as pd

from wonca_rag.config import load_experiment_config
from wonca_rag.paths import PATHS
from wonca_rag.rag.embeddings import (
    build_semantic_index,
    load_semantic_index,
)
from wonca_rag.rag.bm25 import build_bm25_token_corpus
from wonca_rag.rag.retrieval import HybridRetriever, RetrievalSettings

config = load_experiment_config()
CHUNKS_PATH = PATHS.artifacts_root / 'chunks' / 'article_chunks.parquet'
chunks_df = pd.read_parquet(CHUNKS_PATH)

print('Chunks:', len(chunks_df))
print('Embedding model:', config.embeddings.model_name)

Chunks: 2215
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 1. Construir índice semântico

Na primeira execução o modelo de embeddings será baixado e os 2.215 chunks serão codificados. Nas execuções seguintes, o cache será reutilizado se corpus e configuração não mudarem.

In [2]:
semantic_paths = build_semantic_index(
    chunks_df=chunks_df,
    output_dir=PATHS.artifacts_root / 'embeddings',
    model_name=config.embeddings.model_name,
    normalize_embeddings=config.embeddings.normalize_embeddings,
    batch_size=config.embeddings.batch_size,
    force=False,
)

print(semantic_paths)

SemanticIndexPaths(embeddings_npy=WindowsPath('C:/Github/wonca-rag/artifacts/embeddings/chunk_embeddings.npy'), metadata_parquet=WindowsPath('C:/Github/wonca-rag/artifacts/embeddings/chunk_embedding_metadata.parquet'), index_metadata_json=WindowsPath('C:/Github/wonca-rag/artifacts/embeddings/semantic_index_metadata.json'))


## 2. Conferir índice semântico

In [3]:
embedding_matrix, embedding_metadata, semantic_meta = load_semantic_index(
    PATHS.artifacts_root / 'embeddings'
)

print('Shape:', embedding_matrix.shape)
print('Metadata rows:', len(embedding_metadata))
print('Index metadata:', semantic_meta)

Shape: (2215, 384)
Metadata rows: 2215
Index metadata: {'index_version': '1.0.0', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'normalize_embeddings': True, 'embedding_dimension': 384, 'n_chunks': 2215, 'corpus_fingerprint': '9704df7effd81e4617b7b2f1c9bf395510de81f1c227e727bc119aa175801998'}


## 3. Construir corpus BM25

In [4]:
bm25_paths = build_bm25_token_corpus(
    chunks_df=chunks_df,
    output_dir=PATHS.artifacts_root / 'indexes' / 'bm25',
)

for name, path in bm25_paths.items():
    print(name, ':', path)

tokens_parquet : C:\Github\wonca-rag\artifacts\indexes\bm25\bm25_tokens.parquet
metadata_json : C:\Github\wonca-rag\artifacts\indexes\bm25\bm25_index_metadata.json


## 4. Carregar o retriever híbrido

In [5]:
retriever = HybridRetriever(
    chunks_df=chunks_df,
    embeddings_dir=PATHS.artifacts_root / 'embeddings',
    bm25_dir=PATHS.artifacts_root / 'indexes' / 'bm25',
    model_name=config.embeddings.model_name,
    normalize_embeddings=config.embeddings.normalize_embeddings,
    settings=RetrievalSettings(
        semantic_top_k=config.retrieval.semantic_top_k,
        bm25_top_k=config.retrieval.bm25_top_k,
        final_top_k=config.retrieval.final_top_k,
        rrf_k=config.retrieval.rrf_k,
    ),
)

print('HybridRetriever: OK')

HybridRetriever: OK


## 5. Testar retrieval dentro do artigo atual

Escolhemos um artigo com vários chunks apenas para inspeção.

In [6]:
article_counts = chunks_df.groupby('article_id').size().sort_values(ascending=False)
sample_article_id = article_counts.index[0]

query = (
    'primary care continuity coordination first contact '
    'comprehensiveness measurement assessment'
)

current_results = retriever.retrieve_current_article(
    query=query,
    article_id=sample_article_id,
)

print('Article:', sample_article_id)
display(current_results[[
    'hybrid_rank', 'page_start', 'page_end',
    'semantic_rank', 'bm25_rank', 'rrf_score', 'text'
]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Article: ART_01AB8A1A032F


,hybrid_rank,page_start,page_end,semantic_rank,bm25_rank,rrf_score,text
0,1,2,2,8.0,5.0,0.030090,\n *\n\n+\n'  &\n 4!\n  #\n...
1,2,3,3,4.0,11.0,0.029710," !!$\n\n+# $ &\n& $""\n  ..."
2,3,3,3,5.0,10.0,0.029670,'&  $\n\n \n \n\n\n'  ' # \n...
3,4,2,3,9.0,9.0,0.028986, *\n&\n\n\n$\n\n\n  *\n& ' ...
4,5,1,1,NaN,1.0,0.016393,\n\n\n\n\n\n\n\n \n...
5,6,5,5,1.0,NaN,0.016393,\n  !#\n\n \n *\n&\n!$\n...
6,7,1,1,NaN,2.0,0.016129,\n $\n&  & ($\n\n $\n ...
7,8,6,6,2.0,NaN,0.016129,\n  $ &  # &\n($\n...


## 6. Testar retrieval histórico com trava temporal

In [7]:
article_years = (
    chunks_df[['article_id', 'publication_year']]
    .dropna()
    .drop_duplicates()
    .sort_values('publication_year')
    .reset_index(drop=True)
)

sample_row = article_years.iloc[len(article_years) // 2]
current_article_id = sample_row['article_id']
current_year = int(sample_row['publication_year'])

history_results = retriever.retrieve_history(
    query=query,
    current_year=current_year,
    current_article_id=current_article_id,
)

print('Current article:', current_article_id)
print('Current year:', current_year)
display(history_results[[
    'hybrid_rank', 'article_id', 'publication_year',
    'page_start', 'page_end', 'rrf_score', 'text'
]])

if not history_results.empty:
    assert history_results['publication_year'].dropna().astype(int).lt(current_year).all()

print('Temporal retrieval: OK')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Current article: ART_60E33BF801FC
Current year: 1999


,hybrid_rank,article_id,publication_year,page_start,page_end,rrf_score,text
0,1,ART_A7505BC65928,1979,6,7,0.032787,"problems are unrecognized, there will be\neith..."
1,2,ART_C5BDCCAEBF0B,1980,2,3,0.030536,""" is used in different ways by\ndifferent rese..."
2,3,ART_A7505BC65928,1979,3,4,0.030366,than referred into\ncare for the first time. I...
3,4,ART_0AEECF5C220B,1976,1,2,0.030077,"Mepicat, Cane:\nJuly 1976, Vol. XV, No. 7\nCon..."
4,5,ART_8AC4523EBB7F,1996,2,3,0.029412,-\nment rather than on reducing mortality and ...
5,6,ART_2179244B822A,1989,1,2,0.028814,have\nmade judgments about whether or not\nthe...
6,7,ART_A7505BC65928,1979,7,8,0.016129,the population perceive access to be con-\nven...
7,8,ART_2179244B822A,1989,1,1,0.016129,"Hugh D. Allen, MD, Columbus, Ohio\nFredric\nBu..."


Temporal retrieval: OK


## Resultado

Ao final desta etapa teremos dois componentes persistidos:

- `artifacts/embeddings/`, matriz semântica e metadados;
- `artifacts/indexes/bm25/`, corpus lexical tokenizado.

A próxima etapa será definir e validar os prompts e o schema estruturado S/N/D antes de conectar os nove modelos.